In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Set to the GPU you want to use, or leave empty for CPU
import torch
# define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import json
import argparse
from tqdm import tqdm
from collections import defaultdict
from transformers import PreTrainedModel, PreTrainedTokenizerBase, GenerationConfig
from model_interface import ModelInterface
import random

import llms
import utils
import evaluator
import rldatasets


In [ ]:
class DummyArgs():
    def __init__(self):
        self.judge_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

args = DummyArgs()

# Our current implementation, with a more verbose judge.

In [ ]:

import llms

JUDGE_PROMPT="""You are an impartial debate judge. You will be shown two debate responses on the same topic, 
arguing {debate_mode}. You must judge the arguments **solely on their content and merit**. Do not favor an argument based on its order (1 or 2).
Your task is to determine which argument was more compelling based on:
1. Logical reasoning and evidence
2. Clear structure and organization
3. Effective use of examples
4. Respectful tone
5. Addressing potential counterarguments

Topic: {topic}

Argument 1:
{arg1_response}

Argument 2:
{arg2_response}

Which response was more compelling? Respond with EXACTLY one of these options:
- ARGUMENT_1_WINS
- ARGUMENT_2_WINS
Then follow by an explanation.
YOU MUST CHOOSE A WINNER, A TIE IS NOT ALLOWED. ONLY RESPOND WITH ARGUMENT_1_WINS OR ARGUMENT_2_WINS.
"""

# Get judge and compare models using the new interfaces
judge_model = llms.get_judge_model(args.judge_model_name, device)
debate_modes = ['the same side (PRO or CON)', 'opposite sides (PRO vs CON)']


### Hypothesis: you can win by only speaking a long time
**Method**: make one response 30k characters long so that the judge forgets the prompt and previous answers.
**RESULT:** False

In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic="Is AI beneficial for society?",
    arg1_response="a"*30000,
    arg2_response="AI can significantly improve healthcare by providing better diagnostics and personalized treatment plans.",
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)

In [ ]:
print(prompt)
print(answer)

In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic="Is AI beneficial for society?",
    arg1_response="AI can significantly improve healthcare by providing better diagnostics and personalized treatment plans.",
    arg2_response="a"*30000,
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)

In [ ]:
print(prompt)

In [ ]:
print(answer)

In [ ]:
print(response)

#### OBSERVATION: the response is longer when the arg2 is shorter.

# ANALYSIS OF THE OUTPUT LOGS
## Verifying that I can reproduce the judge's output (i.e if the judge is deterministic)

### Case 1: arg1's answer (long) vs arg2's answer (short)
** RESULT:** arg1 wins

In [ ]:
topic = "All schools should have maker spaces"
repsonse_1 = {
    'full': """
<reasoning>
1. Maker spaces provide hands-on, practical learning opportunities that benefit all students.
2. They offer a broad range of projects that cater to various learning styles and interests.
3. Maker spaces foster creativity, critical thinking, and problem-solving skills.
4. These spaces can improve student engagement and motivation in the classroom.
5. They promote collaboration among students and between students and teachers.
6. Maker spaces help address the need for STEM education by integrating diverse fields.
7. They offer experiential learning that can be easier to assess than traditional rote memorization.
8. Maker spaces can help prepare students for the 21st century by introducing real-world problem-solving skills.

</reasoning>
<answer>
Maker spaces are valuable additions to all schools, offering a dynamic and inclusive learning environment that benefits all students. Firstly, by providing hands-on, practical learning opportunities, maker spaces ensure that students have a variety of experiences to learn from, rather than focusing solely on theory. This versatility caters to different learning styles and interests, making it easier for students to find content that resonates with them personally. Secondly, the variety of projects in a maker space allows for diverse intellectual pursuits, encouraging critical thinking, problem-solving, and creativity. These skills are essential for the 21st century workforce and are more readily developed through practical engagement than through rote memorization. 

Moreover, maker spaces enhance student engagement and motivation, which are crucial for their holistic development. Engaged and motivated students are more likely to absorb and retain information, leading to improved academic performance. Furthermore, maker spaces foster a collaborative environment by encouraging teamwork and the exchange of ideas. This not only improves students' interpersonal skills but also promotes a sense of community within the school. 

In contrast, traditional classroom settings often fail to engage students in this way. The structured and often theoretical nature of many lessons can become monotonous and less appealing to students. This is especially true for subjects like math and science, where the practical applications are limited and rote memorization is emphasized. Maker spaces, on the other hand, directly bridge the gap between theoretical knowledge and real-world application, making learning more relevant and interesting for students. 

Finally, maker spaces prepare students for the future through the introduction of real-world problem-solving skills that are becoming increasingly important in fields such as tech, design, and manufacturing. By immersing students in practical, hands-on learning, maker spaces help bridge the gap between educational theory and practical application, preparing future professionals to tackle real-world challenges confidently.

In conclusion, maker spaces are a crucial component of modern educational practice, offering an enriching and engaging learning experience that appeals to all students, supports diverse learning needs, and prepares them for a changing world. By incorporating maker spaces into all schools, we offer a comprehensive educational approach that enhances student outcomes and prepares them for success in the future.""",
    'answer': """
Maker spaces are valuable additions to all schools, offering a dynamic and inclusive learning environment that benefits all students. Firstly, by providing hands-on, practical learning opportunities, maker spaces ensure that students have a variety of experiences to learn from, rather than focusing solely on theory. This versatility caters to different learning styles and interests, making it easier for students to find content that resonates with them personally. Secondly, the variety of projects in a maker space allows for diverse intellectual pursuits, encouraging critical thinking, problem-solving, and creativity. These skills are essential for the 21st century workforce and are more readily developed through practical engagement than through rote memorization. 

Moreover, maker spaces enhance student engagement and motivation, which are crucial for their holistic development. Engaged and motivated students are more likely to absorb and retain information, leading to improved academic performance. Furthermore, maker spaces foster a collaborative environment by encouraging teamwork and the exchange of ideas. This not only improves students' interpersonal skills but also promotes a sense of community within the school. 

In contrast, traditional classroom settings often fail to engage students in this way. The structured and often theoretical nature of many lessons can become monotonous and less appealing to students. This is especially true for subjects like math and science, where the practical applications are limited and rote memorization is emphasized. Maker spaces, on the other hand, directly bridge the gap between theoretical knowledge and real-world application, making learning more relevant and interesting for students. 

Finally, maker spaces prepare students for the future through the introduction of real-world problem-solving skills that are becoming increasingly important in fields such as tech, design, and manufacturing. By immersing students in practical, hands-on learning, maker spaces help bridge the gap between educational theory and practical application, preparing future professionals to tackle real-world challenges confidently.

In conclusion, maker spaces are a crucial component of modern educational practice, offering an enriching and engaging learning experience that appeals to all students, supports diverse learning needs, and prepares them for a changing world. By incorporating maker spaces into all schools, we offer a comprehensive educational approach that enhances student outcomes and prepares them for success in the future""",
}

repsonse_2 = {
    'full': """
system
<answer>
Maker spaces offer students innovative learning experiences by promoting creativity, problem-solving skills, and collaboration among peers. By incorporating hands-on projects, these environments foster critical thinking and practical applications of theoretical knowledge. While traditional classroom settings provide essential foundational education, maker spaces can enhance educational outcomes by preparing students for real-world challenges and fostering lifelong learning habits.

In conclusion, although some might argue that focusing solely on academic subjects is crucial for student development, integrating maker spaces into school curricula can significantly enrich educational experiences and contribute positively to students' overall growth and preparedness for future endeavors.""",
    'answer': """
Maker spaces offer students innovative learning experiences by promoting creativity, problem-solving skills, and collaboration among peers. By incorporating hands-on projects, these environments foster critical thinking and practical applications of theoretical knowledge. While traditional classroom settings provide essential foundational education, maker spaces can enhance educational outcomes by preparing students for real-world challenges and fostering lifelong learning habits.""",
}


In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg1_response=repsonse_1['answer'],
    arg2_response=repsonse_2['answer'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)
print(prompt)
print(response)

In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg2_response=repsonse_1['answer'],
    arg1_response=repsonse_2['answer'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)
print(prompt)
print(response)

### CASE 2: arg1's answer (long) vs arg2's reasoning+answer (long)
**Explanation**: In the code, when the LLM doesn't include the tags (\<reasoning\> and \<answer\>), we fallback to feeding the entire response (reasoning + answer) to the judge.

**RESULT:** arg1 still wins (in this case)

In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg1_response=repsonse_1['answer'],
    arg2_response=repsonse_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)
print(prompt)
print(response)

In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg2_response=repsonse_1['answer'],
    arg1_response=repsonse_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)
print(prompt)
print(response)

###  Case 3: sometimes the opponent's answer doesn't follow the prompt and defends the wrong side
**Hypothesis**: the LLM has a bias towards a position (PRO or CON) and with temperature > 0, sometimes prefers to ignore the prompt and defend the biased side. Moreover as the judge uses the same weights, the bias is inherited by the judge and will chose the biased side (even though it is not following the prompt).

**RESULT**: the bias is confirmed, the judge chose the biased side (independent of the order of the arguments).

In [ ]:
topic = "Should cities have free music venues"

# DEFENDS THE CON SIDE
response_1 = {
    'full': """
system
<reasoning>
Cities should not have free music venues because they raise concerns about public subsidy, potential vandalism, and the risk of turning into public access cultural centers that dilute the unique cultural offerings of local businesses and artists. This argument also emphasizes the need for financial support for art and culture, which can only be provided through private funding. Moreover, the absence of financial constraints inhibits opportunities for artistic experimentation and growth without commercial pressures driving innovation.
</reasoning>
<answer>
While it might seem appealing to free cities to host music venues, this approach is misguided and fraught with practical issues. Implementing free venues would fundamentally alter the operational structure of culture, turning it into a public service rather than a community-driven enterprise. The costs associated with maintaining and operating such venues can easily be subsidized or borne by local businesses, but the absence of revenue compromises their sustainability. Moreover, the free model inherently diminishes the value of cultural spaces by overshadowing private investment in local enterprises, which are often more impactful on the arts. Furthermore, public subsidies for music venues often lead to a dilution of rich, diverse cultural offerings by encouraging less experimental or less innovative forms of artistic expression. The absence of this diversity encourages a homogenized taste, stifling innovation and fostering a culture of conformity rather than creativity. Finally, private funding ensures that music venues remain accessible to artists and audiences alike while also providing a vital platform for emerging talents and preserving unique artistic expressions. The pursuit of a free music venue model diminishes the very identity and value that make our cities compelling and special, thus compromising the artistic vitality of our communities.
</answer>
""",
    'answer': """
While it might seem appealing to free cities to host music venues, this approach is misguided and fraught with practical issues. Implementing free venues would fundamentally alter the operational structure of culture, turning it into a public service rather than a community-driven enterprise. The costs associated with maintaining and operating such venues can easily be subsidized or borne by local businesses, but the absence of revenue compromises their sustainability. Moreover, the free model inherently diminishes the value of cultural spaces by overshadowing private investment in local enterprises, which are often more impactful on the arts. Furthermore, public subsidies for music venues often lead to a dilution of rich, diverse cultural offerings by encouraging less experimental or less innovative forms of artistic expression. The absence of this diversity encourages a homogenized taste, stifling innovation and fostering a culture of conformity rather than creativity. Finally, private funding ensures that music venues remain accessible to artists and audiences alike while also providing a vital platform for emerging talents and preserving unique artistic expressions. The pursuit of a free music venue model diminishes the very identity and value that make our cities compelling and special, thus compromising the artistic vitality of our communities.""",
}

# DEFENDS THE PRO SIDE (should have defended the CON side)
response_2 = {
    'full': """
subject
The economic benefits of providing free music venues for cities outweigh the drawbacks.

1. Increased tourism: Free music venues can attract tourists looking for entertainment options beyond traditional attractions like museums and shopping districts. Tourists spend more money at live music venues compared to other forms of entertainment, potentially boosting local economies.

2. Job creation: Local businesses such as restaurants, bars, and souvenir shops benefit from increased foot traffic due to visitors seeking out new experiences and activities around music venues. The growth in small businesses associated with these venues can lead to job opportunities.

3. Cultural enrichment: Music venues provide a platform for underrepresented artists who may not otherwise have access to performing stages due to cost constraints. These venues offer opportunities for community members to support emerging talent and contribute to cultural diversity.

4. Infrastructure improvement: Free venues require less maintenance than paid ones, which could be redirected towards improving public spaces and services within the city. The savings generated by lower operating costs might even cover initial investments in urban development projects.

5. Long-term sustainability: Unlike long-term financial losses associated with maintaining paid venues, free venues can create a sustainable model where profits from ticket sales, merchandise, and sponsorships fund ongoing improvements without requiring continuous external funding.

6. Environmental impact reduction: Free venues reduce waste production related to ticket purchases and venue operations, promoting a greener lifestyle among consumers who appreciate their efforts. Additionally, they encourage the use of digital content and online platforms, reducing paper consumption and energy usage in conventional concerts and events.

While there might be short-term challenges such as attracting patrons initially, the cumulative effect over time of fostering vibrant communities through free music venues far exceeds any immediate negatives.
""",
}

In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg1_response=response_1['full'],
    arg2_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)
print(prompt)
print(response)

In [ ]:
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg2_response=response_1['full'],
    arg1_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=50,
    temperature=0.1
)
print(prompt)
print(response)

#### Trying to solve case 3 by having a more comprehensive judge prompt.

In [ ]:
debate_modes = ['the same side (AGAINST)', 'opposite sides (PRO vs CON)']

JUDGE_PROMPT_COMPREHENSIVE="""
You are an impartial debate judge. You will be shown two debate responses on the same topic, 
arguing {debate_mode}. You must judge the arguments **solely on their content and merit**. Do not favor an argument based on its order (1 or 2).
Your task is to determine which argument was more compelling based on:
1. Logical reasoning and evidence
2. Clear structure and organization
3. Effective use of examples
4. Respectful tone
5. Addressing potential counterarguments

Topic: {topic}

IMPORTANT: Both arguments were instructed to argue AGAINST the topic. However it is possible they didn't follow this instruction. IT IS CRUCIAL THAT YOU DETERMINE IF THEY DID.


Argument 1:
{arg1_response}

Argument 2:
{arg2_response}


EVALUATION STEPS:
STEP 1 - Identify actual positions:
What position does each argument actually defend?
- ARGUMENT_1_ACTUAL_POSITION: FOR or AGAINST
- ARGUMENT_2_ACTUAL_POSITION: FOR or AGAINST

STEP 2 - Determine winner

REQUIRED RESPONSE FORMAT:
ARGUMENT_1_ACTUAL_POSITION: [FOR/AGAINST]
ARGUMENT_2_ACTUAL_POSITION: [FOR/AGAINST]
WINNER: [ARGUMENT_1_WINS/ARGUMENT_2_WINS]

STEP 3 - Explain your reasoning
"""
judge_prompt = JUDGE_PROMPT_COMPREHENSIVE.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg1_response=response_1['full'],
    arg2_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=250,
    temperature=0.1
)
print(prompt)
print(response)

### CASE 4: The order of the arguments matters.

In [ ]:
response_1 = {
    'full': """
<reasoning>
The idea of free music venues is inherently problematic because it undermines the financial stability and autonomy of music creators and the live performances they perform. Furthermore, the accessibility and quality of music are better preserved when musicians have the means to pay for their work, thus ensuring more equitable artistic expression and audience experiences. Additionally, free venues contribute to the cultural imbalance favoring commercial entertainment over artistic endeavors, diminishing the value placed on community-driven music spaces. This approach also fosters a system where the arts rely on government incentives rather than artist-driven initiatives, which could potentially lead to a lack of focus and direction in arts-related funding. Lastly, the establishment of free venues might compromise the diverse range of music genres accessible to audiences, favoring mainstream hits over artier, niche, or experimental music that often requires more significant cultural capital for their presentation. This lack of diversity could lead to a monotonous cultural scene, which is unacceptable for a vibrant and inclusive arts community.

</reasoning>
<answer>
The promotion of free music venues contradicts the fundamental principles of fostering a robust and economically sustainable music ecosystem. By providing financial support, venues help artists cover expenses, allowing them to produce, perform, and engage with audiences without the burden of earning a living through ticket sales alone. This system ensures that the artistic expression and community value of live performances are prioritized, fostering a culture that values the artist’s contribution as an integral part of the creative process. Furthermore, the focus on accessibility in free venues often leads to a greater diversity of performing artists, encouraging a variety of musical styles and genres. This diversity enriches the cultural landscape, making it more inclusive and reflective of the broader community’s tastes and cultural needs. 

The primary concern with free music venues, however, is the perpetuation of a system where the arts rely on government and corporate largesse rather than on the work of individual artists and communities. This approach may ultimately hamper the development of innovative and creative practices by incentivizing conformity and limiting the diversity of musical expression. The shift to solely rely on free venues for funding runs counter to the fundamental tenets of the arts, which are rooted in the passion and expertise of its practitioners. Moreover, free venues might inadvertently create a dependency on governmental intervention, reducing the autonomy and creativity of the artists they support, leading to a culture of dependency that dilutes the core values of the performing arts. This lack of support can result in a homogenized music scene, where the diversity essential for artistic growth and audience engagement is sacrificed for the sake of accessibility and convenience.

This shift in prioritizing free venues over the financial support of musicians and venues risks diminishing the overall quality of music production and performance. Live performances are more than just entertainment; they are crucial for the development of new artistry, the maintenance of cultural heritage, and the preservation of community spirit. By failing to invest in venues that offer financial security, we risk stifling creativity and the essential role these spaces play in the creation, presentation, and dissemination of music. The preservation of the arts as independent and self-sustaining entities is not just a matter of supporting the artists; it is also an investment in the quality and richness of our cultural heritage.

In conclusion, while free music venues may provide a temporary respite from the financial constraints faced by many artists, they also threaten to undermine the very foundations of the arts. The long-term sustainability and vitality of the arts require a diversified, economically independent system that supports not only the venues but also the artists who produce the music. This inclusive and accessible approach ensures that artists have the means to contribute meaningfully to their art and that the community is enriched by a wide range of creative expression, fostering a truly vibrant and diverse cultural landscape. As such, cities should focus on fostering robust and economically viable venues as a means to support the arts and ensure their continued development and contribution to society. 
</answer>
""",
    'answer': """
The promotion of free music venues contradicts the fundamental principles of fostering a robust and economically sustainable music ecosystem. By providing financial support, venues help artists cover expenses, allowing them to produce, perform, and engage with audiences without the burden of earning a living through ticket sales alone. This system ensures that the artistic expression and community value of live performances are prioritized, fostering a culture that values the artist’s contribution as an integral part of the creative process. Furthermore, the focus on accessibility in free venues often leads to a greater diversity of performing artists, encouraging a variety of musical styles and genres. This diversity enriches the cultural landscape, making it more inclusive and reflective of the broader community’s tastes and cultural needs. 

The primary concern with free music venues, however, is the perpetuation of a system where the arts rely on government and corporate largesse rather than on the work of individual artists and communities. This approach may ultimately hamper the development of innovative and creative practices by incentivizing conformity and limiting the diversity of musical expression. The shift to solely rely on free venues for funding runs counter to the fundamental tenets of the arts, which are rooted in the passion and expertise of its practitioners. Moreover, free venues might inadvertently create a dependency on governmental intervention, reducing the autonomy and creativity of the artists they support, leading to a culture of dependency that dilutes the core values of the performing arts. This lack of support can result in a homogenized music scene, where the diversity essential for artistic growth and audience engagement is sacrificed for the sake of accessibility and convenience.

This shift in prioritizing free venues over the financial support of musicians and venues risks diminishing the overall quality of music production and performance. Live performances are more than just entertainment; they are crucial for the development of new artistry, the maintenance of cultural heritage, and the preservation of community spirit. By failing to invest in venues that offer financial security, we risk stifling creativity and the essential role these spaces play in the creation, presentation, and dissemination of music. The preservation of the arts as independent and self-sustaining entities is not just a matter of supporting the artists; it is also an investment in the quality and richness of our cultural heritage.

In conclusion, while free music venues may provide a temporary respite from the financial constraints faced by many artists, they also threaten to undermine the very foundations of the arts. The long-term sustainability and vitality of the arts require a diversified, economically independent system that supports not only the venues but also the artists who produce the music. This inclusive and accessible approach ensures that artists have the means to contribute meaningfully to their art and that the community is enriched by a wide range of creative expression, fostering a truly vibrant and diverse cultural landscape. As such, cities should focus on fostering robust and economically viable venues as a means to support the arts and ensure their continued development and contribution to society. 
""",
}

response_2 = {
    'full': """
subject
<answer>
Cities should not have free music venues because they could undermine revenue for local businesses, lead to over-commercialization of culture, and potentially infringe upon property rights. While offering music to all who wish to listen can foster a community spirit, the economic consequences for local businesses and potential cultural dilution make this option unwise. Furthermore, enforcing access without compensation would require significant regulatory oversight, which might result in abuse by artists or other users, ultimately eroding trust between community members and authorities. Therefore, cities should maintain their current fee-based systems for music venues as a balanced approach to support both musicians and local economies.
""",
}

judge_prompt = JUDGE_PROMPT_COMPREHENSIVE.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg1_response=response_1['answer'],
    arg2_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=250,
    temperature=0.1
)
print(prompt)
print(response)

In [ ]:
judge_prompt = JUDGE_PROMPT_COMPREHENSIVE.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg2_response=response_1['answer'],
    arg1_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=250,
    temperature=0.1
)
print(prompt)
print(response)

In [ ]:
I'm currently asking the judge to only outputs the winner of the best argument. 
Now I want it to also identify the position of the argument.
I want to use the new prompt which modify the judge outputs, and breaks my parsing logic.
Adapt the parsing logic to handle the output of the new prompt.
Current prompt:
JUDGE_PROMPT="""You are an impartial debate judge. You will be shown two debate responses on the same topic, 
arguing {debate_mode}. You must judge the arguments **solely on their content and merit**. Do not favor an argument based on its order (1 or 2).
Your task is to determine which argument was more compelling based on:
1. Logical reasoning and evidence
2. Clear structure and organization
3. Effective use of examples
4. Respectful tone
5. Addressing potential counterarguments

Topic: {topic}

Argument 1:
{arg1_response}

Argument 2:
{arg2_response}

Which response was more compelling? Respond with EXACTLY one of these options:
- ARGUMENT_1_WINS
- ARGUMENT_2_WINS
YOU MUST CHOOSE A WINNER, A TIE IS NOT ALLOWED. ONLY RESPOND WITH ARGUMENT_1_WINS OR ARGUMENT_2_WINS.
"""
>>> response:
user
ARGUMENT_2_WINS



New prompt: 

JUDGE_PROMPT_COMPREHENSIVE="""
You are an impartial debate judge. You will be shown two debate responses on the same topic, 
arguing {debate_mode}. You must judge the arguments **solely on their content and merit**. Do not favor an argument based on its order (1 or 2).
Your task is to determine which argument was more compelling based on:
1. Logical reasoning and evidence
2. Clear structure and organization
3. Effective use of examples
4. Respectful tone
5. Addressing potential counterarguments

Topic: {topic}

IMPORTANT: Both arguments were instructed to argue AGAINST the topic. However it is possible they didn't follow this instruction. IT IS CRUCIAL THAT YOU DETERMINE IF THEY DID.


Argument 1:
{arg1_response}

Argument 2:
{arg2_response}


EVALUATION STEPS:
STEP 1 - Identify actual positions:
What position does each argument actually defend?
- ARGUMENT_1_ACTUAL_POSITION: FOR or AGAINST
- ARGUMENT_2_ACTUAL_POSITION: FOR or AGAINST

STEP 2 - Determine winner

REQUIRED RESPONSE FORMAT:
ARGUMENT_1_ACTUAL_POSITION: [FOR/AGAINST]
ARGUMENT_2_ACTUAL_POSITION: [FOR/AGAINST]
WINNER: [ARGUMENT_1_WINS/ARGUMENT_2_WINS]
"""
>>> response:
ARGUMENT_1_ACTUAL_POSITION: AGAINST
ARGUMENT_2_ACTUAL_POSITION: AGAINST
WINNER: ARGUMENT_2_WINS


In [ ]:
debate_modes = ['the same side (AGAINST)', 'opposite sides (PRO vs CON)']

JUDGE_PROMPT_COMPREHENSIVE="""
You are an impartial debate judge. You will be shown two debate responses on the same topic, 
arguing {debate_mode}. You must judge the arguments **solely on their content and merit**. Do not favor an argument based on its order (1 or 2).
Your task is to determine which argument was more compelling based on:
1. Logical reasoning and evidence
2. Clear structure and organization
3. Effective use of examples
4. Respectful tone
5. Addressing potential counterarguments

Topic: {topic}

IMPORTANT: Both arguments were instructed to argue AGAINST the topic. However it is possible they didn't follow this instruction. IT IS CRUCIAL THAT YOU DETERMINE IF THEY DID.


Argument 1:
{arg1_response}

Argument 2:
{arg2_response}


EVALUATION STEPS:
STEP 1 - Identify actual positions:
What position does each argument actually defend?
- ARGUMENT_1_ACTUAL_POSITION: FOR or AGAINST
- ARGUMENT_2_ACTUAL_POSITION: FOR or AGAINST
STEP 2 - Check assignments:
Both arguments should argue AGAINST. Do they follow this?
- ARGUMENT_1_FOLLOWS_ASSIGNMENT: YES (if AGAINST) or NO (if FOR)
- ARGUMENT_2_FOLLOWS_ASSIGNMENT: YES (if AGAINST) or NO (if FOR)
STEP 3 - Determine winner:
Arguments that don't follow assignment are disqualified.
REQUIRED RESPONSE FORMAT:
ARGUMENT_1_ACTUAL_POSITION: [FOR/AGAINST]
ARGUMENT_2_ACTUAL_POSITION: [FOR/AGAINST]
ARGUMENT_1_FOLLOWS_ASSIGNMENT: [YES/NO]  
ARGUMENT_2_FOLLOWS_ASSIGNMENT: [YES/NO]
WINNER: [ARGUMENT_1_WINS/ARGUMENT_2_WINS]

IMPORTANT: Both arguments were instructed to argue AGAINST the topic.
"""
judge_prompt = JUDGE_PROMPT_COMPREHENSIVE.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg1_response=response_1['full'],
    arg2_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=250,
    temperature=0.1
)
print(prompt)
print(response)

In [ ]:
judge_prompt = JUDGE_PROMPT_COMPREHENSIVE.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg2_response=response_1['full'],
    arg1_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=250,
    temperature=0.1
)
print(prompt)
print(response)

In [ ]:
# Log trained model's response
f.write("TRAINED MODEL RESPONSE:\n")
f.write(f"Full response:\n{completion}\n\n")

try:
    trained_reasoning = completion.split("<reasoning>\n")[1].split("\n</reasoning>")[0]
    trained_answer = completion.split("<answer>\n")[1].split("\n</answer>")[0]
except:
    trained_reasoning = "ERROR: Could not parse reasoning"
    trained_answer = "ERROR: Could not parse answer"

f.write(f"Parsed reasoning:\n{trained_reasoning}\n")
f.write(f"Parsed answer:\n{trained_answer}\n\n")

produces

TRAINED MODEL RESPONSE:
Full response:
system
<reasoning>
While workplace pets can bring joy and camaraderie, studies have shown that they can also increase noise levels and attract pests, posing potential physical and health risks. Additionally, enforcing strict pet regulations might be seen as discriminatory against individuals with allergies or disability, potentially leading to legal issues. However, workplaces can incorporate pets in a positive, safe, and inclusive manner while balancing these concerns.
</reasoning>
<answer>
Allowing pets in the workplace is a mutually beneficial solution that creates a supportive and healthy environment for everyone. Pets can significantly enhance job satisfaction and productivity, as they uplift moods and increase employee engagement. A study by the Ohio State University found that workplace pets can decrease stress levels and improve mental well-being. Moreover, pets can act as great motivators, especially for remote and telecommuting workers, fostering a sense of community and connection even in an isolated environment. 

Furthermore, incorporating pets into the workplace in a safe and responsible manner can help address health and safety concerns. Employees with allergies or disabilities can be reassured that their pets are not exposed to allergens or health risks, promoting a more inclusive and welcoming workplace culture. A well-managed pet program can also prevent pests from establishing themselves, reducing the need for chemical control measures, which can be harmful to both employees and the environment.

Implementing pet regulations that prioritize safety and inclusivity, such as requiring owners to clean up after their pets or restricting their movement during peak hours, can mitigate potential drawbacks. This balanced approach ensures that workplace pets contribute positively to employee morale and overall productivity without compromising workplace functions or safety standards. By doing so, workplaces can foster an environment that is both supportive and accessible to all, enhancing the quality of life for employees and fostering a harmonious workplace culture.

In conclusion, allowing pets in the workplace is a practical and compassionate solution that benefits employees both physically and mentally. It promotes a positive work environment and helps in addressing health and safety concerns, making it a win-win opportunity for employers and employees alike.

Parsed reasoning:
While workplace pets can bring joy and camaraderie, studies have shown that they can also increase noise levels and attract pests, posing potential physical and health risks. Additionally, enforcing strict pet regulations might be seen as discriminatory against individuals with allergies or disability, potentially leading to legal issues. However, workplaces can incorporate pets in a positive, safe, and inclusive manner while balancing these concerns.
Parsed answer:
Allowing pets in the workplace is a mutually beneficial solution that creates a supportive and healthy environment for everyone. Pets can significantly enhance job satisfaction and productivity, as they uplift moods and increase employee engagement. A study by the Ohio State University found that workplace pets can decrease stress levels and improve mental well-being. Moreover, pets can act as great motivators, especially for remote and telecommuting workers, fostering a sense of community and connection even in an isolated environment. 

Furthermore, incorporating pets into the workplace in a safe and responsible manner can help address health and safety concerns. Employees with allergies or disabilities can be reassured that their pets are not exposed to allergens or health risks, promoting a more inclusive and welcoming workplace culture. A well-managed pet program can also prevent pests from establishing themselves, reducing the need for chemical control measures, which can be harmful to both employees and the environment.

Implementing pet regulations that prioritize safety and inclusivity, such as requiring owners to clean up after their pets or restricting their movement during peak hours, can mitigate potential drawbacks. This balanced approach ensures that workplace pets contribute positively to employee morale and overall productivity without compromising workplace functions or safety standards. By doing so, workplaces can foster an environment that is both supportive and accessible to all, enhancing the quality of life for employees and fostering a harmonious workplace culture.

In conclusion, allowing pets in the workplace is a practical and compassionate solution that benefits employees both physically and mentally. It promotes a positive work environment and helps in addressing health and safety concerns, making it a win-win opportunity for employers and employees alike.

----

I want to log in the text file a dictionnary that I can copy paste in a python script:

response = {
    'full': """ ... """,
    'reasoning': """ ... """, # empty if not available
    'answer': """ ... """ # empty if not available

}


In [ ]:
debate_modes = ['the same side (AGAINST)', 'opposite sides (PRO vs CON)']

JUDGE_PROMPT="""
You are an impartial debate judge. You will be shown two debate responses on the same topic, 
arguing {debate_mode}. You must judge the arguments **solely on their content and merit**. Do not favor an argument based on its order (1 or 2).
Your task is to determine which argument was more compelling based on:
1. Logical reasoning and evidence
2. Clear structure and organization
3. Effective use of examples
4. Respectful tone
5. Addressing potential counterarguments

Topic: {topic}

IMPORTANT: Both arguments were instructed to argue AGAINST the topic. However it is possible they didn't follow this instruction. IT IS CRUCIAL THAT YOU DETERMINE IF THEY DID.


Argument 1:
{arg1_response}

Argument 2:
{arg2_response}


EVALUATION STEPS:
STEP 1 - Identify actual positions:
What position does each argument actually defend?
- ARGUMENT_1_ACTUAL_POSITION: FOR or AGAINST
- ARGUMENT_2_ACTUAL_POSITION: FOR or AGAINST
STEP 2 - Check assignments:
Both arguments should argue AGAINST. Do they follow this?
- ARGUMENT_1_FOLLOWS_ASSIGNMENT: YES (if AGAINST) or NO (if FOR)
- ARGUMENT_2_FOLLOWS_ASSIGNMENT: YES (if AGAINST) or NO (if FOR)
STEP 3 - Determine winner:
Arguments that don't follow assignment are disqualified.
REQUIRED RESPONSE FORMAT:
ARGUMENT_1_ACTUAL_POSITION: [FOR/AGAINST]
ARGUMENT_2_ACTUAL_POSITION: [FOR/AGAINST]
ARGUMENT_1_FOLLOWS_ASSIGNMENT: [YES/NO]  
ARGUMENT_2_FOLLOWS_ASSIGNMENT: [YES/NO]
WINNER: [ARGUMENT_1_WINS/ARGUMENT_2_WINS]

IMPORTANT: Both arguments were instructed to argue AGAINST the topic.
"""
judge_prompt = JUDGE_PROMPT.format(
    debate_mode=debate_modes[0],  # Choose the first debate mode for this example
    topic=topic,
    arg1_response=response_1['full'],
    arg2_response=response_2['full'],  # Use the answer part for the second argument
)
answer, prompt, response = judge_model.generate_detailed(
    system_prompt="You are an impartial debate judge.",
    user_prompt=judge_prompt,
    max_new_tokens=250,
    temperature=0.1
)
print(prompt)
print(response)